<a href="https://colab.research.google.com/github/srishanthdevoju/Celebal_Internship/blob/main/week7_srishanthdevoju.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DocuMind — RAG Document Question Answering System

A **Retrieval-Augmented Generation (RAG)** system that answers questions based on your custom documents.  
Upload PDFs, text files, or markdown documents and ask intelligent questions — the AI retrieves relevant information and generates accurate, grounded answers.

---

## Architecture Overview

```
Upload PDF/TXT --- FastAPI Backend --- Text Extraction --- Chunking
                                                              |
                                                              |
                                              Embedding (all-MiniLM-L6-v2)
                                                              |
                                                              |
                                               ChromaDB Vector Store

Ask Question --- Query Embedding --- Similarity Search --- Top-K Chunks
                                                              |
                                                              |
                                           Groq LLM (Llama 3.3 70B)
                                                              |
                                                              |
                                           Generated Answer + Sources
```


---

## 1. Install Dependencies

In [1]:
!pip install fastapi uvicorn[standard] python-dotenv pypdf2 sentence-transformers chromadb groq langchain-text-splitters python-multipart python-docx -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

---

## 2. Configuration

All system-wide settings: paths, API keys, model names, chunking parameters, and retrieval settings.

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv
from google.colab import userdata

load_dotenv()

BASE_DIR = Path(".").resolve()
DATA_DIR = BASE_DIR / "data"
CHROMA_DB_DIR = BASE_DIR / "chroma_db"

DATA_DIR.mkdir(exist_ok=True)
CHROMA_DB_DIR.mkdir(exist_ok=True)

try:
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except userdata.SecretNotFoundError:
    GROQ_API_KEY = ""

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
EMBEDDING_DIMENSION = 384

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

GROQ_MODEL = "llama-3.3-70b-versatile"
GROQ_MAX_TOKENS = 1024
GROQ_TEMPERATURE = 0.3

TOP_K_RESULTS = 5

ALLOWED_EXTENSIONS = {".pdf", ".txt", ".md", ".docx"}
MAX_FILE_SIZE_MB = 50

print(" Configuration loaded successfully")
print(f"   Base directory   : {BASE_DIR}")
print(f"   Data directory   : {DATA_DIR}")
print(f"   ChromaDB path    : {CHROMA_DB_DIR}")
print(f"   Embedding model  : {EMBEDDING_MODEL_NAME} (dim={EMBEDDING_DIMENSION})")
print(f"   LLM model        : {GROQ_MODEL}")
print(f"   Chunk size       : {CHUNK_SIZE} chars, overlap {CHUNK_OVERLAP}")
print(f"   Top-K retrieval  : {TOP_K_RESULTS}")
print(f"   GROQ API Key     : {'Set' if GROQ_API_KEY and GROQ_API_KEY != 'your_groq_api_key_here' else 'NOT SET'}")

 Configuration loaded successfully
   Base directory   : /content
   Data directory   : /content/data
   ChromaDB path    : /content/chroma_db
   Embedding model  : all-MiniLM-L6-v2 (dim=384)
   LLM model        : llama-3.3-70b-versatile
   Chunk size       : 1000 chars, overlap 200
   Top-K retrieval  : 5
   GROQ API Key     : Set


---

## 3. Pydantic Models (API Schemas)

Define the request and response schemas used by the REST API.

In [3]:
from pydantic import BaseModel, Field
from typing import Optional


class UploadResponse(BaseModel):
    filename: str
    document_id: str
    chunk_count: int
    status: str = "success"
    message: str = ""

class QueryRequest(BaseModel):
    question: str = Field(..., min_length=1, max_length=2000)

class SourceChunk(BaseModel):
    text: str
    page: Optional[int] = None
    source: str = ""
    relevance_score: float = 0.0


class QueryResponse(BaseModel):
    answer: str
    sources: list[SourceChunk] = []
    question: str
    status: str = "success"


class DocumentInfo(BaseModel):
    document_id: str
    filename: str
    chunk_count: int
    file_type: str
    file_size: str


class HealthResponse(BaseModel):
    status: str = "healthy"
    embedding_model: str = ""
    vector_store: str = "chromadb"
    llm: str = ""
    documents_loaded: int = 0


print(" Pydantic models defined: UploadResponse, QueryRequest, SourceChunk, QueryResponse, DocumentInfo, HealthResponse")

 Pydantic models defined: UploadResponse, QueryRequest, SourceChunk, QueryResponse, DocumentInfo, HealthResponse


---

## 4. RAG Pipeline — Document Ingestion

This module handles:
1. **Loading** documents — extracting text from PDFs (page-by-page via PyPDF2) or reading plain text/markdown files.
2. **Chunking** — splitting text into overlapping chunks using LangChain's `RecursiveCharacterTextSplitter` so each chunk fits the embedding model's context window and retains semantic coherence.

In [4]:
import logging
from PyPDF2 import PdfReader
from docx import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import datasets

logger_ingestion = logging.getLogger("ingestion")


def load_pdf(filepath):
    filepath = Path(filepath)
    reader = PdfReader(str(filepath))
    pages = []

    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if text and text.strip():
            pages.append({
                "text": text.strip(),
                "page": i + 1,
                "source": filepath.name,
            })

    logger_ingestion.info(f"Loaded {len(pages)} pages from PDF: {filepath.name}")
    return pages


def load_docx(filepath):
    filepath = Path(filepath)
    document = Document(filepath)
    full_text = []
    for para in document.paragraphs:
        full_text.append(para.text)
    text = "\n".join(full_text)

    if not text.strip():
        return []

    logger_ingestion.info(f"Loaded DOCX file: {filepath.name} ({len(text)} chars)")
    return [{
        "text": text.strip(),
        "page": None,
        "source": filepath.name,
    }]


def load_text(filepath):
    filepath = Path(filepath)
    text = filepath.read_text(encoding="utf-8", errors="ignore")

    if not text.strip():
        return []

    logger_ingestion.info(f"Loaded text file: {filepath.name} ({len(text)} chars)")
    return [{
        "text": text.strip(),
        "page": None,
        "source": filepath.name,
    }]


def load_document(filepath):
    filepath = Path(filepath)
    ext = filepath.suffix.lower()

    if ext not in ALLOWED_EXTENSIONS:
        raise ValueError(f"Unsupported file type: {ext}. Allowed: {ALLOWED_EXTENSIONS}")

    if ext == ".pdf":
        return load_pdf(filepath)
    elif ext == ".docx":
        return load_docx(filepath)
    else:
        return load_text(filepath)


def chunk_documents(documents, chunk_size=None, chunk_overlap=None):
    chunk_size = chunk_size or CHUNK_SIZE
    chunk_overlap = chunk_overlap or CHUNK_OVERLAP

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", ". ", " ", ""],
    )

    chunks = []
    chunk_index = 0

    for doc in documents:
        text_chunks = splitter.split_text(doc["text"])
        for text_chunk in text_chunks:
            chunks.append({
                "text": text_chunk,
                "page": doc.get("page"),
                "source": doc.get("source", ""),
                "chunk_index": chunk_index,
            })
            chunk_index += 1

    logger_ingestion.info(
        f"Split {len(documents)} document sections into {len(chunks)} chunks "
        f"(size={chunk_size}, overlap={chunk_overlap})"
    )
    return chunks


def ingest_file(filepath):
    documents = load_document(filepath)
    if not documents:
        raise ValueError(f"No text content could be extracted from: {filepath}")

    chunks = chunk_documents(documents)
    logger_ingestion.info(f"Ingested {filepath}: {len(chunks)} chunks ready for embedding")
    return chunks

print(" Ingestion module defined: load_pdf, load_docx, load_text, load_document, chunk_documents, ingest_file")

 Ingestion module defined: load_pdf, load_docx, load_text, load_document, chunk_documents, ingest_file


In [5]:
def load_huggingface_archive(dataset_name, text_column, split="train", limit=5):
    try:
        print(f" Loading Hugging Face dataset: {dataset_name}, split: {split}, limit: {limit}...")
        dataset = datasets.load_dataset(dataset_name, split=split)

        documents = []
        for i, row in enumerate(dataset):
            if i >= limit:
                break
            if text_column in row:
                documents.append({
                    "text": str(row[text_column]),
                    "page": None,
                    "source": f"hf://{dataset_name}/{split}/{i}",
                })
        print(f" Loaded {len(documents)} documents from Hugging Face dataset '{dataset_name}'.")
        return documents
    except Exception as e:
        print(f" Error loading Hugging Face dataset '{dataset_name}': {e}")
        return []


print(" `load_huggingface_archive` function defined.")

 `load_huggingface_archive` function defined.


In [6]:
from google.colab import files
import shutil

def upload_and_ingest_documents():
    print("Please upload your documents (PDF, TXT, DOCX, MD). Press 'Done' when finished.")
    uploaded = files.upload()

    if not uploaded:
        print("No files uploaded.\n")
        return

    successful_uploads = []
    for filename, content in uploaded.items():
        original_filepath = Path(filename)
        ext = original_filepath.suffix.lower()

        if ext not in ALLOWED_EXTENSIONS:
            print(f" Skipping {filename}: Unsupported file type {ext}. Allowed: {ALLOWED_EXTENSIONS}")
            continue

        filepath = DATA_DIR / original_filepath.name
        counter = 1
        while filepath.exists():
            filepath = DATA_DIR / f"{original_filepath.stem}_{counter}{ext}"
            counter += 1

        with open(filepath, 'wb') as f:
            f.write(content)
        print(f" Uploaded and saved: {filepath.name}")

        try:
            chunks = ingest_file(filepath)
            doc_id = add_documents(chunks)
            successful_uploads.append({
                "filename": filepath.name,
                "document_id": doc_id,
                "chunk_count": len(chunks)
            })
            print(f"   Ingested {len(chunks)} chunks from '{filepath.name}' (ID: {doc_id})")
        except Exception as e:
            print(f" Failed to ingest {filepath.name}: {e}")
            if filepath.exists():
                filepath.unlink()

    print("\n--- Upload and Ingestion Summary ---")
    if successful_uploads:
        for upload_info in successful_uploads:
            print(f" {upload_info['filename']} (ID: {upload_info['document_id']}, Chunks: {upload_info['chunk_count']})")
        print("All uploaded documents processed.")
    else:
        print("No documents were successfully uploaded and ingested.")

---

## 5. RAG Pipeline — Embedding Module

This module wraps **sentence-transformers** (`all-MiniLM-L6-v2`) to convert text into 384-dimensional vector representations.  
Uses a **singleton pattern** to load the model only once and reuse it across calls.

> **Note:** The first run downloads the model (~80 MB) automatically.

In [7]:
from sentence_transformers import SentenceTransformer

logger_embeddings = logging.getLogger("embeddings")

_embedding_model = None

def _get_embedding_model():
    global _embedding_model
    if _embedding_model is None:
        logger_embeddings.info(f"Loading embedding model: {EMBEDDING_MODEL_NAME}...")
        _embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
        logger_embeddings.info(f"Embedding model loaded successfully.")
    return _embedding_model


def get_embeddings(texts):
    if not texts:
        return []

    model = _get_embedding_model()
    embeddings = model.encode(texts, show_progress_bar=False, convert_to_numpy=True)

    logger_embeddings.info(f"Generated {len(embeddings)} embeddings (dim={embeddings.shape[1]})")
    return embeddings.tolist()


def get_single_embedding(text):
    model = _get_embedding_model()
    embedding = model.encode([text], show_progress_bar=False, convert_to_numpy=True)
    return embedding[0].tolist()


print(" Embedding module defined: get_embeddings, get_single_embedding")

 Embedding module defined: get_embeddings, get_single_embedding


### 5.1 Demo — Generate Embeddings

Let's load the model and generate embeddings for sample text to see the output dimensions.

In [8]:
import numpy as np

sample_texts = [
    "Retrieval-Augmented Generation combines search with language models.",
    "ChromaDB is a vector database for storing embeddings.",
    "The weather is sunny today.",
]

print(" Loading embedding model (first run downloads ~80MB)...")
sample_embeddings = get_embeddings(sample_texts)

print(f"\n Generated {len(sample_embeddings)} embeddings")
print(f"   Embedding dimension: {len(sample_embeddings[0])}")
print(f"   First 10 values of embedding 0: {sample_embeddings[0][:10]}")

emb = np.array(sample_embeddings)
norms = np.linalg.norm(emb, axis=1, keepdims=True)
cos_sim = (emb @ emb.T) / (norms @ norms.T)

print("\n Cosine Similarity Matrix:")
for i, t1 in enumerate(sample_texts):
    for j, t2 in enumerate(sample_texts):
        if j >= i:
            print(f"   [{i}] vs [{j}]: {cos_sim[i][j]:.4f}  (\"{t1[:40]}...\" vs \"{t2[:40]}...\")")

print("\n Notice: Sentences 0 & 1 (both about AI/tech) are more similar than sentence 2 (weather).")

 Loading embedding model (first run downloads ~80MB)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


 Generated 3 embeddings
   Embedding dimension: 384
   First 10 values of embedding 0: [-0.0605340301990509, -0.03613980486989021, -0.017261764034628868, 0.08305316418409348, -0.021899741142988205, 0.11021460592746735, 0.005437697749584913, -0.007171297445893288, 0.026867171749472618, -0.05892856791615486]

 Cosine Similarity Matrix:
   [0] vs [0]: 1.0000  ("Retrieval-Augmented Generation combines ..." vs "Retrieval-Augmented Generation combines ...")
   [0] vs [1]: 0.2451  ("Retrieval-Augmented Generation combines ..." vs "ChromaDB is a vector database for storin...")
   [0] vs [2]: -0.0507  ("Retrieval-Augmented Generation combines ..." vs "The weather is sunny today....")
   [1] vs [1]: 1.0000  ("ChromaDB is a vector database for storin..." vs "ChromaDB is a vector database for storin...")
   [1] vs [2]: -0.0323  ("ChromaDB is a vector database for storin..." vs "The weather is sunny today....")
   [2] vs [2]: 1.0000  ("The weather is sunny today...." vs "The weather is sunny today

---

## 6. RAG Pipeline — Vector Store (ChromaDB)

This module manages all ChromaDB operations:
- **Adding documents** — embedding chunks and storing them in named collections
- **Querying** — searching across all collections for the most relevant chunks
- **Listing** — enumerating all ingested documents
- **Deleting** — removing documents from the store

Each uploaded document gets its own ChromaDB collection, named after the source file.

In [9]:
import uuid
import chromadb
from chromadb.config import Settings

logger_vectorstore = logging.getLogger("vectorstore")

_chroma_client = None


def _get_chroma_client():
    global _chroma_client
    if _chroma_client is None:
        logger_vectorstore.info(f"Initializing ChromaDB at: {CHROMA_DB_DIR}")
        _chroma_client = chromadb.PersistentClient(
            path=str(CHROMA_DB_DIR),
            settings=Settings(anonymized_telemetry=False),
        )
        logger_vectorstore.info("ChromaDB client initialized.")
    return _chroma_client


def _sanitize_collection_name(name):
    sanitized = name.rsplit(".", 1)[0]
    sanitized = "".join(c if c.isalnum() or c in "_-" else "_" for c in sanitized)
    sanitized = sanitized.strip("_-")

    if len(sanitized) < 3:
        sanitized = sanitized + "_doc"
    if len(sanitized) > 63:
        sanitized = sanitized[:63]

    if not sanitized[0].isalnum():
        sanitized = "d" + sanitized[1:]
    if not sanitized[-1].isalnum():
        sanitized = sanitized[:-1] + "d"

    return sanitized


def add_documents(chunks, document_id=None):
    if not chunks:
        raise ValueError("No chunks to add.")

    client = _get_chroma_client()

    source_name = chunks[0].get("source", "document")
    if document_id is None:
        document_id = _sanitize_collection_name(source_name)

    texts = [chunk["text"] for chunk in chunks]
    embeddings = get_embeddings(texts)

    collection = client.get_or_create_collection(
        name=document_id,
        metadata={"source_file_name": source_name},
    )

    ids = [f"{document_id}_{i}" for i in range(len(chunks))]
    metadatas = [
        {
            "source_file_name": chunk.get("source", ""),
            "document_collection_id": document_id,
            "page": chunk.get("page") or 0,
            "chunk_index": chunk.get("chunk_index", i),
        }
        for i, chunk in enumerate(chunks)
    ]

    collection.add(
        ids=ids,
        embeddings=embeddings,
        documents=texts,
        metadatas=metadatas,
    )

    logger_vectorstore.info(f"Added {len(chunks)} chunks to collection '{document_id}'")
    return document_id


def vector_query(question, n_results=5):
    client = _get_chroma_client()
    collections = client.list_collections()

    if not collections:
        logger_vectorstore.warning("No collections found in the vector store.")
        return []

    query_embedding = get_single_embedding(question)
    all_results = []

    for collection_obj in collections:
        collection = client.get_collection(collection_obj.name)

        if collection.count() == 0:
            continue

        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=min(n_results, collection.count()),
            include=["documents", "metadatas", "distances"],
        )

        if results["documents"] and results["documents"][0]:
            for i, doc in enumerate(results["documents"][0]):
                distance = results["distances"][0][i] if results["distances"] else 0
                metadata = results["metadatas"][0][i] if results["metadatas"] else {}

                similarity = 1 / (1 + distance)

                all_results.append({
                    "text": doc,
                    "page": metadata.get("page"),
                    "source": metadata.get("source_file_name", ""),
                    "document_collection_id": metadata.get("document_collection_id"),
                    "score": similarity,
                })

    all_results.sort(key=lambda x: x["score"], reverse=True)
    return all_results[:n_results]


def list_documents():
    client = _get_chroma_client()
    collections = client.list_collections()

    documents = []
    for collection_obj in collections:
        collection = client.get_collection(collection_obj.name)
        metadata = collection.metadata or {}
        documents.append({
            "document_id": collection_obj.name,
            "source_file_name": metadata.get("source_file_name", collection_obj.name),
            "chunk_count": collection.count(),
        })

    return documents


def delete_document(document_id):
    client = _get_chroma_client()

    try:
        client.delete_collection(document_id)
        logger_vectorstore.info(f"Deleted collection: {document_id}")
        return True
    except ValueError:
        logger_vectorstore.warning(f"Collection not found: {document_id}")
        return False


def get_collection_count():
    client = _get_chroma_client()
    return len(client.list_collections())


print(" Vector store module defined: add_documents, vector_query, list_documents, delete_document, get_collection_count")

 Vector store module defined: add_documents, vector_query, list_documents, delete_document, get_collection_count


---

## 7. RAG Pipeline — Answer Generation (Groq LLM)

This module uses the **Groq API** with **Llama 3.3 70B** to generate answers grounded in retrieved context.

Key design choices:
- **System prompt** instructs the LLM to answer ONLY from the provided context (preventing hallucination).
- **Low temperature** (0.3) ensures factual, consistent answers.
- Retrieved chunks are formatted with source attribution in the prompt.

In [10]:
from groq import Groq

logger_generation = logging.getLogger("generation")

_groq_client = None


def _get_groq_client():
    global _groq_client
    if _groq_client is None:
        if not GROQ_API_KEY or GROQ_API_KEY == "your_groq_api_key_here":
            raise ValueError(
                "Groq API key not configured. "
                "Please set GROQ_API_KEY in your .env file. "
                "Get a free key at https://console.groq.com"
            )
        _groq_client = Groq(api_key=GROQ_API_KEY)
        logger_generation.info("Groq client initialized.")
    return _groq_client


RAG_SYSTEM_PROMPT = """You are an intelligent document assistant. Your role is to answer questions accurately based ONLY on the provided context from the user's documents.

Rules:
1. Answer the question using ONLY the information in the provided context.
2. If the context does not contain enough information to answer the question, clearly state: "I don't have enough information in the uploaded documents to answer this question."
3. Be concise but thorough in your answers.
4. When relevant, mention which source document the information comes from.
5. Do not make up information or use knowledge outside the provided context.
6. Use clear formatting with bullet points or numbered lists when appropriate."""


def _build_prompt(question, context_chunks):
    context_parts = []
    for i, chunk in enumerate(context_chunks, 1):
        source = chunk.get("source", "Unknown")
        page = chunk.get("page")
        page_info = f" (Page {page})" if page else ""
        context_parts.append(
            f"--- Context {i} [Source: {source}{page_info}] ---\n{chunk['text']}"
        )

    context_str = "\n\n".join(context_parts)

    return f"""Based on the following context from the uploaded documents, answer the question.

{context_str}

--- Question ---
{question}

--- Answer ---"""


def generate_answer(question, context_chunks):
    if not context_chunks:
        return "No relevant documents were found to answer your question. Please upload documents first."

    client = _get_groq_client()
    prompt = _build_prompt(question, context_chunks)

    logger_generation.info(f"Generating answer for: '{question}' with {len(context_chunks)} context chunks")

    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": RAG_SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        max_tokens=GROQ_MAX_TOKENS,
        temperature=GROQ_TEMPERATURE,
        top_p=0.9,
    )

    answer = response.choices[0].message.content.strip()
    logger_generation.info(f"Answer generated ({len(answer)} chars)")
    return answer


print(" Generation module defined: generate_answer")

 Generation module defined: generate_answer


In [14]:
upload_and_ingest_documents()

Please upload your documents (PDF, TXT, DOCX, MD). Press 'Done' when finished.


Saving DEVOJU SRISHANTH_Doc (5).pdf to DEVOJU SRISHANTH_Doc (5).pdf
 Uploaded and saved: DEVOJU SRISHANTH_Doc (5).pdf
   Ingested 2 chunks from 'DEVOJU SRISHANTH_Doc (5).pdf' (ID: DEVOJU_SRISHANTH_Doc__5)

--- Upload and Ingestion Summary ---
 DEVOJU SRISHANTH_Doc (5).pdf (ID: DEVOJU_SRISHANTH_Doc__5, Chunks: 2)
All uploaded documents processed.


---

## 8. End-to-End RAG Demo

Now that documents are uploaded and ingested, let's run the complete RAG pipeline with a sample question.

In [15]:
print("="*60)
print("  DocuMind — End-to-End RAG Pipeline Demo")
print("="*60)

all_docs = list_documents()

if not all_docs:
    print("\n No documents found in the vector store. Please upload documents first.")
else:
    print(f"\n Documents in vector store ({len(all_docs)}):")
    for doc in all_docs:
        print(f"   - {doc['source_file_name']}  ({doc['chunk_count']} chunks)")

    while True:
        question = input("\n Enter your question (or type 'exit' to quit): ")
        if question.lower() == 'exit':
            break
        if not question.strip():
            print("Please enter a question.")
            continue

        print(f"\n Step 4 — Querying: \"{question}\"")
        retrieved = vector_query(question)
        print(f"   Retrieved {len(retrieved)} relevant chunks")

        for i, r in enumerate(retrieved):
            print(f"\n   Chunk {i+1} (score: {r['score']:.4f}, source: {r['source']}, page: {r.get('page')})")
            print(f"      \"{r['text'][:150]}...\"")

        print(f"\n Step 5 — Generating answer with {GROQ_MODEL}...")
        answer = generate_answer(question, retrieved)
        print(f"\n{'─'*60}")
        print(f" ANSWER:\n")
        print(answer)
        print(f"{'─'*60}")


  DocuMind — End-to-End RAG Pipeline Demo

 Documents in vector store (1):
   - DEVOJU SRISHANTH_Doc (5).pdf  (2 chunks)

 Enter your question (or type 'exit' to quit): does he know spark?

 Step 4 — Querying: "does he know spark?"
   Retrieved 2 relevant chunks

   Chunk 1 (score: 0.3587, source: DEVOJU SRISHANTH_Doc (5).pdf, page: 1)
      "–Analyzed traffic conditions and identified potential hazards
–Enabled real-time alerts for unsafe driving and accident scenarios
TECHNICAL SKILLS
•Pr..."

   Chunk 2 (score: 0.3412, source: DEVOJU SRISHANTH_Doc (5).pdf, page: 1)
      "Srishanth Devoju
Hyderabad, Telangana
♂phone+91-9346328356/envel⌢pesrishanthdevoju5@gmail.com/linkedinLinkedIn/githubGitHub
EDUCATION
CVR College of E..."

 Step 5 — Generating answer with llama-3.3-70b-versatile...

────────────────────────────────────────────────────────────
 ANSWER:

Yes, according to the context, Srishanth Devoju knows Spark. This information is mentioned in the TECHNICAL SKILLS section of Cont

In [16]:
while True:
    question = input("\n Enter your question (or type 'exit' to quit): ")
    if question.lower() == 'exit':
        break
    if not question.strip():
        print("Please enter a question.")
        continue

    print(f" Question: \"{question}\"\n")

    retrieved = vector_query(question)
    print(f" Retrieved {len(retrieved)} chunks (top scores: {[round(r['score'], 4) for r in retrieved]})\n")

    answer = generate_answer(question, retrieved)

    print(f"{'─'*60}")
    print(f" ANSWER:\n")
    print(answer)
    print(f"{'─'*60}")

    print(f"\n Sources:")
    for i, r in enumerate(retrieved):
        page_info = f", page {r['page']}" if r.get('page') else ""
        print(f"   {i+1}. {r['source']}{page_info} (relevance: {r['score']*100:.1f}%)")



 Enter your question (or type 'exit' to quit): what is his cgpa?
 Question: "what is his cgpa?"

 Retrieved 2 chunks (top scores: [0.3823, 0.3448])

────────────────────────────────────────────────────────────
 ANSWER:

His CGPA is 8.52, as mentioned in the "EDUCATION" section of the context from the document "DEVOJU SRISHANTH_Doc (5).pdf (Page 1)". Specifically, it is stated that he has a "B.Tech in CSE (Data Science) – CGPA: 8.52" from CVR College of Engineering.
────────────────────────────────────────────────────────────

 Sources:
   1. DEVOJU SRISHANTH_Doc (5).pdf, page 1 (relevance: 38.2%)
   2. DEVOJU SRISHANTH_Doc (5).pdf, page 1 (relevance: 34.5%)

 Enter your question (or type 'exit' to quit): exit


---

## 9. System Optimizations Experiments

Let's explore some common RAG optimizations to improve retrieval quality and relevance.

In [17]:
import uuid

def run_chunk_adjustment_experiment(document_path, query, new_chunk_size, new_chunk_overlap):
    print(f"\n--- Running Chunk Border Adjustment Experiment ---")
    print(f"Document: {document_path.name}")
    print(f"Query: \"{query}\" ")
    print(f"New Chunking: size={new_chunk_size}, overlap={new_chunk_overlap}")

    temp_doc_uuid = uuid.uuid4().hex[:8]
    temp_doc_id = _sanitize_collection_name(f"{document_path.stem}_{temp_doc_uuid}")

    try:
        print(f" Ingesting '{document_path.name}' into temporary collection '{temp_doc_id}'...")
        raw_documents = load_document(document_path)
        temp_chunks = chunk_documents(raw_documents, chunk_size=new_chunk_size, chunk_overlap=new_chunk_overlap)
        add_documents(temp_chunks, document_id=temp_doc_id)
        print(f" Ingested {len(temp_chunks)} chunks for chunking experiment.")

        print(f" Querying all collections for '{query}'...")
        retrieved_results = vector_query(query, n_results=TOP_K_RESULTS * 2)

        print(f"\n--- Retrieved Context (Top {TOP_K_RESULTS} from temporary collection only) ---")
        relevant_temp_results = [r for r in retrieved_results if r.get('document_collection_id') == temp_doc_id]

        if relevant_temp_results:
            relevant_temp_results.sort(key=lambda x: x['score'], reverse=True)
            print(f"Found {len(relevant_temp_results)} relevant chunks from temporary collection '{temp_doc_id}':")
            for i, r in enumerate(relevant_temp_results[:TOP_K_RESULTS]):
                page_info = f", page {r['page']}" if r.get('page') else ""
                print(f"   [{i+1}] (Score: {r['score']:.4f}, Source: {r['source']}{page_info})")
                print(f"       Text: \"{r['text'][:200]}...\"")
        else:
            print(f"No relevant chunks retrieved from temporary collection '{temp_doc_id}'. This might mean the new chunking strategy didn't yield top results for this query, or the query is not specific enough.")

    except Exception as e:
        print(f" Error during chunk adjustment experiment: {e}")
    finally:
        if delete_document(temp_doc_id):
            print(f"Cleaned up temporary collection '{temp_doc_id}'.")


sample_document_path = next(DATA_DIR.iterdir(), None)

if sample_document_path:
    test_query = "What are the main technical skills?"
    new_chunk_size = 500
    new_chunk_overlap = 100
    run_chunk_adjustment_experiment(sample_document_path, test_query, new_chunk_size, new_chunk_overlap)
else:
    print(" No documents found in DATA_DIR to run the chunk adjustment experiment. Please upload one.")


--- Running Chunk Border Adjustment Experiment ---
Document: DEVOJU SRISHANTH_Doc (5).pdf
Query: "What are the main technical skills?" 
New Chunking: size=500, overlap=100
 Ingesting 'DEVOJU SRISHANTH_Doc (5).pdf' into temporary collection 'DEVOJU_SRISHANTH_Doc__5__fd266c3f'...
 Ingested 5 chunks for chunking experiment.
 Querying all collections for 'What are the main technical skills?'...

--- Retrieved Context (Top 5 from temporary collection only) ---
Found 5 relevant chunks from temporary collection 'DEVOJU_SRISHANTH_Doc__5__fd266c3f':
   [1] (Score: 0.4463, Source: DEVOJU SRISHANTH_Doc (5).pdf, page 1)
       Text: "–Analyzed traffic conditions and identified potential hazards
–Enabled real-time alerts for unsafe driving and accident scenarios
TECHNICAL SKILLS
•Programming Languages:Python, Java, JavaScript, SQL
..."
   [2] (Score: 0.4041, Source: DEVOJU SRISHANTH_Doc (5).pdf, page 1)
       Text: "•Databases:MySQL, MongoDB
•Tools:Git, GitHub, VS Code, AWS Fundamentals, Docker F

### 9.2 Experiment 2: Hybrid Lookup Strategy

In [18]:
import collections
import re

def simple_sparse_search(query, documents, n_results=5):
    query_words = set(re.findall(r'\b\w+\b', query.lower()))
    scored_documents = []

    for doc in documents:
        doc_text = doc['text'].lower()
        doc_words = set(re.findall(r'\b\w+\b', doc_text))
        common_words = query_words.intersection(doc_words)
        score = len(common_words)

        if score > 0:
            scored_documents.append({
                "text": doc['text'],
                "page": doc.get('page'),
                "source": doc.get('source'),
                "score": score,
                "type": "sparse"
            })

    scored_documents.sort(key=lambda x: x['score'], reverse=True)
    return scored_documents[:n_results]

def hybrid_query(question, n_results=TOP_K_RESULTS, sparse_weight=0.3, dense_weight=0.7):
    print(f" Running Hybrid Query for: \"{question}\"")

    dense_results = vector_query(question, n_results=n_results * 2)
    for r in dense_results: r['type'] = "dense"

    client = _get_chroma_client()
    all_chroma_collections = client.list_collections()
    all_available_chunks = []
    for collection_obj in all_chroma_collections:
        collection = client.get_collection(collection_obj.name)
        if collection.count() > 0:
            results = collection.get(
                ids=collection.get()['ids'],
                include=['documents', 'metadatas']
            )
            for i in range(len(results['ids'])):
                all_available_chunks.append({
                    'text': results['documents'][i],
                    'page': results['metadatas'][i].get('page'),
                    'source': results['metadatas'][i].get('source')
                })

    sparse_results = simple_sparse_search(question, all_available_chunks, n_results=n_results * 2)

    combined_results_map = collections.OrderedDict()

    for res in dense_results:
        key = (res['text'], res.get('source'), res.get('page'))
        if key not in combined_results_map:
            combined_results_map[key] = res
            combined_results_map[key]['hybrid_score'] = res['score'] * dense_weight

    for res in sparse_results:
        key = (res['text'], res.get('source'), res.get('page'))
        if key not in combined_results_map:
            combined_results_map[key] = res
            combined_results_map[key]['hybrid_score'] = res['score'] * sparse_weight
        else:
            combined_results_map[key]['hybrid_score'] += res['score'] * sparse_weight

    final_results = list(combined_results_map.values())
    final_results.sort(key=lambda x: x.get('hybrid_score', 0), reverse=True)

    print(f" Retrieved {len(final_results)} combined chunks.")
    return final_results[:n_results]

if list_documents():
    print("\n--- Running Hybrid Lookup Experiment ---")
    hybrid_question = "What technologies are used?"
    hybrid_retrieved = hybrid_query(hybrid_question, n_results=TOP_K_RESULTS)

    print(f"\n--- Hybrid Retrieved Context (Top {TOP_K_RESULTS}) ---")
    if hybrid_retrieved:
        for i, r in enumerate(hybrid_retrieved):
            page_info = f", page {r['page']}" if r.get('page') else ""
            print(f"   [{i+1}] (Score: {r.get('hybrid_score', r['score']):.4f}, Type: {r['type']}, Source: {r['source']}{page_info})")
            print(f"       Text: \"{r['text'][:200]}...\"")

        print("\n--- Generating Answer with Hybrid Context ---")
        hybrid_answer = generate_answer(hybrid_question, hybrid_retrieved)
        print(f"\n{'─'*60}")
        print(f" ANSWER (Hybrid Retrieval):\n")
        print(hybrid_answer)
        print(f"{'─'*60}")
    else:
        print("No chunks retrieved by hybrid search.")
else:
    print(" No documents found in vector store to run the hybrid lookup experiment. Please upload some first.")


--- Running Hybrid Lookup Experiment ---
 Running Hybrid Query for: "What technologies are used?"
 Retrieved 3 combined chunks.

--- Hybrid Retrieved Context (Top 5) ---
   [1] (Score: 0.3000, Type: sparse, Source: None, page 1)
       Text: "–Analyzed traffic conditions and identified potential hazards
–Enabled real-time alerts for unsafe driving and accident scenarios
TECHNICAL SKILLS
•Programming Languages:Python, Java, JavaScript, SQL
..."
   [2] (Score: 0.2707, Type: dense, Source: DEVOJU SRISHANTH_Doc (5).pdf, page 1)
       Text: "–Analyzed traffic conditions and identified potential hazards
–Enabled real-time alerts for unsafe driving and accident scenarios
TECHNICAL SKILLS
•Programming Languages:Python, Java, JavaScript, SQL
..."
   [3] (Score: 0.2605, Type: dense, Source: DEVOJU SRISHANTH_Doc (5).pdf, page 1)
       Text: "Srishanth Devoju
Hyderabad, Telangana
♂phone+91-9346328356/envel⌢pesrishanthdevoju5@gmail.com/linkedinLinkedIn/githubGitHub
EDUCATION
CVR College of Engin

### 9.3 Experiment 3: Re-ranking Layer


In [19]:
from sentence_transformers import CrossEncoder

RE_RANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
_reranker_model = None

def _get_reranker_model():
    global _reranker_model
    if _reranker_model is None:
        print(f" Loading re-ranker model: {RE_RANKER_MODEL_NAME}...")
        _reranker_model = CrossEncoder(RE_RANKER_MODEL_NAME)
        print(f" Re-ranker model loaded successfully.")
    return _reranker_model

def re_rank_chunks(question, retrieved_chunks, top_n=3):
    if not retrieved_chunks:
        return []

    reranker = _get_reranker_model()

    sentence_pairs = [[question, chunk['text']] for chunk in retrieved_chunks]

    scores = reranker.predict(sentence_pairs)

    for i, chunk in enumerate(retrieved_chunks):
        chunk['score'] = float(scores[i])

    retrieved_chunks.sort(key=lambda x: x['score'], reverse=True)

    print(f" Re-ranked {len(retrieved_chunks)} chunks, returning top {top_n}.")
    return retrieved_chunks[:top_n]

if list_documents():
    print("\n--- Running Re-ranking Layer Experiment ---")
    rerank_question = "What are the projects mentioned?"

    initial_retrieved = vector_query(rerank_question, n_results=10)

    if initial_retrieved:
        print(f"Initial retrieval returned {len(initial_retrieved)} chunks (top scores: {[round(r['score'], 4) for r in initial_retrieved]})")

        top_3_reranked = re_rank_chunks(rerank_question, initial_retrieved, top_n=3)

        if top_3_reranked:
            print(f"\n--- Re-ranked Context (Top 3) ---")
            for i, r in enumerate(top_3_reranked):
                page_info = f", page {r['page']}" if r.get('page') else ""
                print(f"   [{i+1}] (Re-rank Score: {r['score']:.4f}, Source: {r['source']}{page_info})")
                print(f"       Text: \"{r['text'][:200]}...\"")

            print("\n--- Generating Answer with Re-ranked Context ---")
            rerank_answer = generate_answer(rerank_question, top_3_reranked)
            print(f"\n{'─'*60}")
            print(f" ANSWER (Re-ranked Retrieval):\n")
            print(rerank_answer)
            print(f"{'─'*60}")
        else:
            print("No chunks remained after re-ranking.")
    else:
        print("Initial retrieval returned no chunks.")
else:
    print(" No documents found in vector store to run the re-ranking experiment. Please upload some first.")


--- Running Re-ranking Layer Experiment ---
Initial retrieval returned 2 chunks (top scores: [0.4106, 0.3742])
 Loading re-ranker model: cross-encoder/ms-marco-MiniLM-L-6-v2...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

 Re-ranker model loaded successfully.
 Re-ranked 2 chunks, returning top 3.

--- Re-ranked Context (Top 3) ---
   [1] (Re-rank Score: -8.6194, Source: DEVOJU SRISHANTH_Doc (5).pdf, page 1)
       Text: "Srishanth Devoju
Hyderabad, Telangana
♂phone+91-9346328356/envel⌢pesrishanthdevoju5@gmail.com/linkedinLinkedIn/githubGitHub
EDUCATION
CVR College of Engineering2023 – 2027
B.Tech in CSE (Data Science)..."
   [2] (Re-rank Score: -10.5621, Source: DEVOJU SRISHANTH_Doc (5).pdf, page 1)
       Text: "–Analyzed traffic conditions and identified potential hazards
–Enabled real-time alerts for unsafe driving and accident scenarios
TECHNICAL SKILLS
•Programming Languages:Python, Java, JavaScript, SQL
..."

--- Generating Answer with Re-ranked Context ---

────────────────────────────────────────────────────────────
 ANSWER (Re-ranked Retrieval):

Based on the provided context from the uploaded documents, the projects mentioned are:

1. **Railway Management System** (2026) - a full-stack railway

###NOTE :PLEASE SET YOUR OWN GROQ API KEY TO TEST